# X1-REAL-1 Colab CUDA external runner

## Scope

This notebook runs the source-bound `x_factor.x1_external_runner` in a separate CUDA process after the X1 preflight, subject, release, split, basis, and prediction gates pass. It produces immutable **raw external execution receipts** for the frozen task-by-intervention matrix. It does not train or update weights, generate cohorts, fit the predictor, create gold labels, reveal outcomes, create verifier artifacts, commit reveal, or run the scientific analysis. A successful run is raw model execution only and is not a scientific result.

TFG has been clarified as CUDA GPU. CPU and TPU/XLA are rejected rather than silently substituted. No package installation, weight/data download, checkpoint deletion, or output reset occurs. When no source is attached, the notebook makes one pinned public source clone of `codex/x-factor-real`; set `X1_AUTO_CLONE_SOURCE=0` to disable that network action. Keep large inputs and outputs on the local `/content` filesystem rather than a mounted Google Drive; Colab documents Drive I/O and quota failures for large or frequently accessed files.

## Required attached inputs

Attach a clean X1-REAL-1 source release and an artifact directory containing the frozen protocol/preflight and custody artifacts. The source root must contain `pyproject.toml`, `x_factor/x1_real_1.py`, and `x_factor/x1_external_runner.py`. The checkpoint, config, tokenizer, registry, and release root must be attached separately. When unambiguous, the notebook auto-discovers one checkpoint, config, tokenizer, and artifact directory below `/content`; multiple matches are reported as blockers rather than guessed. The checkpoint is never copied into the notebook output.

Set overrides before the first code cell when needed:

```python
import os
os.environ['X1_SOURCE_ROOT'] = '/content/x1-real-1-source'
os.environ['X1_ARTIFACT_ROOT'] = '/content/x1-real-1-artifacts'
os.environ['X1_CHECKPOINT'] = '/content/checkpoints/research-subject.pt'
os.environ['X1_MODEL_CONFIG'] = '/content/checkpoints/model-config.json'
os.environ['X1_TOKENIZER_ARTIFACT'] = '/content/checkpoints/tokenizer.json'
os.environ['X1_REGISTRY'] = '/content/x1-real-1-artifacts/registry.json'
os.environ['X1_RELEASE_ROOT'] = '/content/x1-real-1-source'
os.environ['X1_CHECKPOINT_ROOT'] = '/content/checkpoints'
os.environ['X1_RUN_ROOT'] = '/content/x1-runs/operator-selected-run'
os.environ['X1_MAX_NEW_TOKENS'] = '32'
os.environ['X1_DEVICE'] = 'cuda'
os.environ['RUN_EXTERNAL_CUDA'] = '1'
os.environ['X1_AUTHORIZATION'] = 'I AUTHORIZE X1-REAL-1 EXTERNAL CUDA COMPUTE'
os.environ['RUN_CUDA_INTAKE'] = '1'
os.environ['RUN_CUDA_CANARY'] = '1'
os.environ['X1_CANARY_AUTHORIZATION'] = 'I AUTHORIZE X1-REAL-1 CUDA CANARY'
```

`RUN_CUDA_INTAKE` defaults to enabled and hashes an attached candidate without loading it. `RUN_EXTERNAL_CUDA` defaults to disabled. The exact authorization phrase is required again by the runner; the notebook never supplies it automatically. The stable artifact names are `candidate_intake.json`, `preflight.json`, `subject_manifest.json`, `registry.json`, `release_manifest.json`, `primary_split.json`, `tasks.json` or `public_tasks.json`, `basis_qualification.json`, `basis_split.json`, `development_split.json`, `prediction_receipt.json`, and `prediction_commit.json`. Missing or ambiguous files are reported as blockers.

The intake cell hashes an attached candidate without loading it and runs automatically when a checkpoint is discovered. The optional canary cell validates a blocked candidate's strict checkpoint/runtime identity on CUDA without promotion, training, or scientific claims. Run all cells in order. Reusing the same `X1_RUN_ROOT` resumes immutable receipts; it never deletes or overwrites evidence. The final ZIP contains receipts and metadata only, not checkpoint weights.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
from datetime import datetime, timezone
from pathlib import Path
import platform
import re
import shutil
import subprocess
import sys
from typing import Any

if sys.version_info < (3, 10):
    raise RuntimeError('X1 Colab requires Python 3.10 or newer; runtime=' + platform.python_version())
CONTENT_ROOT = Path('/content').resolve()
CONTENT_ROOT.mkdir(parents=True, exist_ok=True)
SAFE_TOKEN = re.compile(r'[A-Za-z0-9][A-Za-z0-9._-]{0,95}')
CUDA_DEVICE = re.compile(r'cuda(?::[0-9]+)?')
HEX64 = re.compile(r'[0-9a-f]{64}')
AUTHORIZATION_PHRASE = 'I AUTHORIZE X1-REAL-1 EXTERNAL CUDA COMPUTE'
BLOCKERS: list[str] = []


def add_blocker(value: str) -> None:
    if value not in BLOCKERS:
        BLOCKERS.append(value)


def env_path(name: str) -> Path | None:
    value = os.environ.get(name, '').strip()
    return Path(value).expanduser().resolve() if value else None


def strict_pairs(pairs: list[tuple[str, Any]]) -> dict[str, Any]:
    result: dict[str, Any] = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f'duplicate JSON object key: {key!r}')
        result[key] = value
    return result


def reject_constant(value: str) -> None:
    raise ValueError(f'nonstandard JSON constant is forbidden: {value}')


def strict_json_loads(text: str, label: str) -> Any:
    try:
        return json.loads(text, object_pairs_hook=strict_pairs, parse_constant=reject_constant)
    except (json.JSONDecodeError, ValueError) as exc:
        raise ValueError(f'{label}: {exc}') from exc


def strict_json_load(path: str | Path) -> Any:
    target = Path(path)
    return strict_json_loads(target.read_text(encoding='utf-8'), str(target))


def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()


def write_json_no_overwrite(path: str | Path, value: Any) -> bool:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    payload = (json.dumps(value, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False) + '\n').encode('utf-8')
    if target.exists():
        return False
    with target.open('xb') as handle:
        handle.write(payload)
        handle.flush()
        os.fsync(handle.fileno())
    return True


def persist_or_load(path: str | Path, value: Any) -> tuple[Any, bool]:
    target = Path(path)
    if target.exists():
        return strict_json_load(target), True
    write_json_no_overwrite(target, value)
    return value, False


def discover_candidates(suffixes: set[str], names: set[str]) -> list[Path]:
    matches: list[Path] = []
    for current, directories, filenames in os.walk(CONTENT_ROOT, followlinks=False):
        current_path = Path(current)
        directories[:] = sorted(name for name in directories if name not in {'.git', '__pycache__', 'x1_runs'} and not (current_path / name).is_symlink())
        for filename in sorted(filenames):
            path = current_path / filename
            if path.is_symlink() or not path.is_file():
                continue
            if path.suffix.lower() in suffixes or filename in names:
                if not path.is_relative_to(RUN_ROOT):
                    matches.append(path.resolve())
    return sorted(set(matches))


def choose_discovered(current: Path | None, suffixes: set[str], names: set[str], label: str) -> Path | None:
    if current is not None:
        return current
    matches = discover_candidates(suffixes, names)
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        add_blocker(f'AMBIGUOUS_AUTO_DISCOVERED_{label}: ' + ', '.join(str(path) for path in matches[:8]))
    return None

def valid_run_id(value: str) -> bool:
    return bool(SAFE_TOKEN.fullmatch(value))


requested_id = os.environ.get('X1_RUN_ID', '').strip()
if requested_id and not valid_run_id(requested_id):
    add_blocker('X1_RUN_ID contains unsafe characters')
    requested_id = ''
run_id = requested_id or datetime.now(timezone.utc).strftime('x1-cuda-%Y%m%dT%H%M%SZ')
run_override = env_path('X1_RUN_ROOT')
if run_override is not None:
    RUN_ROOT = run_override
    if requested_id and RUN_ROOT.name != requested_id:
        add_blocker('X1_RUN_ID must equal X1_RUN_ROOT final component')
    RUN_ID = requested_id or RUN_ROOT.name
else:
    RUN_ID = run_id
    RUN_ROOT = (CONTENT_ROOT / 'x1_runs' / RUN_ID).resolve()
RUN_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT = env_path('X1_ARTIFACT_ROOT')
CHECKPOINT_PATH = env_path('X1_CHECKPOINT')
CONFIG_PATH = env_path('X1_MODEL_CONFIG')
TOKENIZER_PATH = env_path('X1_TOKENIZER_ARTIFACT')
CHECKPOINT_PATH = choose_discovered(CHECKPOINT_PATH, {'.pt', '.pth', '.ckpt', '.safetensors'}, set(), 'CHECKPOINT')
CONFIG_PATH = choose_discovered(CONFIG_PATH, set(), {'model-config.json', 'config.json'}, 'CONFIG')
TOKENIZER_PATH = choose_discovered(TOKENIZER_PATH, set(), {'tokenizer.json', 'tokenizer_v4_32k.json'}, 'TOKENIZER')
if ARTIFACT_ROOT is None:
    artifact_candidates = []
    for current, directories, filenames in os.walk(CONTENT_ROOT, followlinks=False):
        current_path = Path(current)
        directories[:] = sorted(name for name in directories if name not in {'.git', '__pycache__', 'x1_runs'} and not (current_path / name).is_symlink())
        if 'preflight.json' in filenames or 'candidate_intake.json' in filenames:
            artifact_candidates.append(current_path.resolve())
    if len(artifact_candidates) == 1:
        ARTIFACT_ROOT = artifact_candidates[0]
    elif len(artifact_candidates) > 1:
        add_blocker('AMBIGUOUS_AUTO_DISCOVERED_ARTIFACT_ROOT: ' + ', '.join(str(path) for path in artifact_candidates[:8]))

REGISTRY_OVERRIDE = env_path('X1_REGISTRY')
RELEASE_ROOT = env_path('X1_RELEASE_ROOT')
CHECKPOINT_ROOT = env_path('X1_CHECKPOINT_ROOT')
DEVICE = os.environ.get('X1_DEVICE', 'cuda').strip() or 'cuda'
if not CUDA_DEVICE.fullmatch(DEVICE):
    add_blocker('X1_DEVICE must be cuda or cuda:N; CPU and TPU are unsupported')
try:
    MAX_NEW_TOKENS = int(os.environ.get('X1_MAX_NEW_TOKENS', '32'))
except ValueError:
    MAX_NEW_TOKENS = 32
    add_blocker('X1_MAX_NEW_TOKENS must be an integer')
if not 1 <= MAX_NEW_TOKENS <= 2048:
    add_blocker('X1_MAX_NEW_TOKENS must be between 1 and 2048')
if CHECKPOINT_PATH is not None and CHECKPOINT_PATH.is_file() and CHECKPOINT_PATH.is_relative_to(RUN_ROOT):
    add_blocker('checkpoint must not be inside the packaged run root')
    CHECKPOINT_PATH = None
if ARTIFACT_ROOT is not None and (ARTIFACT_ROOT == RUN_ROOT or RUN_ROOT.is_relative_to(ARTIFACT_ROOT)):
    add_blocker('artifact root and run root must not overlap')
print(json.dumps({'run_id': RUN_ID, 'run_root': str(RUN_ROOT), 'device': DEVICE, 'max_new_tokens': MAX_NEW_TOKENS, 'blockers': BLOCKERS}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import importlib
import os
from pathlib import Path
import shutil
import subprocess
import sys
from typing import Any

STABLE_ARTIFACTS = (
    'candidate_intake.json',
    'preflight.json',
    'subject_manifest.json',
    'registry.json',
    'release_manifest.json',
    'primary_split.json',
    'tasks.json',
    'public_tasks.json',
    'basis_qualification.json',
    'basis_split.json',
    'development_split.json',
    'prediction_receipt.json',
    'prediction_commit.json',
)


def walk_error(error: OSError) -> None:
    add_blocker(f'ARTIFACT_WALK_ERROR:{error}')

AUTO_CLONE_SOURCE = os.environ.get('X1_AUTO_CLONE_SOURCE', '1').strip() == '1'
SOURCE_GIT_URL = os.environ.get('X1_SOURCE_GIT_URL', 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git').strip()
SOURCE_GIT_REF = os.environ.get('X1_SOURCE_GIT_REF', 'codex/x-factor-real').strip()
def source_tree(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / 'pyproject.toml').is_file()
        and (path / 'x_factor' / 'x1_real_1.py').is_file()
        and (path / 'x_factor' / 'x1_external_runner.py').is_file()
        and (path / 'x_factor' / 'x1_cuda_canary.py').is_file()
    )


source_override = env_path('X1_SOURCE_ROOT')
if source_override is None and AUTO_CLONE_SOURCE:
    clone_path = CONTENT_ROOT / 'x1-real-1-source'
    if clone_path.exists():
        if not source_tree(clone_path):
            add_blocker(f'AUTO_CLONE_TARGET_EXISTS_BUT_INVALID:{clone_path}')
    else:
        git_executable = shutil.which('git')
        if git_executable is None:
            add_blocker('GIT_IS_UNAVAILABLE_FOR_SOURCE_CLONE')
        else:
            clone_result = subprocess.run([git_executable, 'clone', '--depth', '1', '--single-branch', '--branch', SOURCE_GIT_REF, SOURCE_GIT_URL, str(clone_path)], capture_output=True, text=True, check=False, timeout=300)
            if clone_result.returncode != 0:
                detail = (clone_result.stderr or clone_result.stdout or '').strip()[-1200:]
                add_blocker(f'SOURCE_CLONE_FAILED:{detail}')
            elif not source_tree(clone_path):
                add_blocker(f'SOURCE_CLONE_INVALID:{clone_path}')

if source_override is not None:
    source_candidates = [source_override]
else:
    source_candidates = [CONTENT_ROOT]
    if CONTENT_ROOT.is_dir():
        source_candidates.extend(sorted(path.resolve() for path in CONTENT_ROOT.iterdir() if path.is_dir()))
valid_sources = [path for path in source_candidates if source_tree(path)]
SOURCE_REVISION = None
if source_override is not None and not valid_sources:
    add_blocker(f'X1_SOURCE_ROOT is not a complete X1 source release: {source_override}')
elif source_override is None and len(valid_sources) != 1:
    add_blocker('set X1_SOURCE_ROOT to the one attached source release; discovery is absent or ambiguous')
SOURCE_ROOT = valid_sources[0] if len(valid_sources) == 1 else None
if SOURCE_ROOT is not None:
    try:
        SOURCE_REVISION = subprocess.run(['git', '-C', str(SOURCE_ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=False, timeout=30).stdout.strip() or None
    except (OSError, subprocess.SubprocessError):
        SOURCE_REVISION = None
PROTOCOL_PATH = SOURCE_ROOT / 'x_factor' / 'protocols' / 'x1_real_1_v1.json' if SOURCE_ROOT else None
RUNNER_SOURCE = SOURCE_ROOT / 'x_factor' / 'x1_external_runner.py' if SOURCE_ROOT else None

if ARTIFACT_ROOT is not None and not ARTIFACT_ROOT.is_dir():
    add_blocker(f'X1_ARTIFACT_ROOT is not a directory: {ARTIFACT_ROOT}')
    ARTIFACT_ROOT = None
ARTIFACT_PATHS: dict[str, Path] = {}
ARTIFACT_MATCHES: dict[str, list[Path]] = {}
if ARTIFACT_ROOT is not None:
    all_files = []
    for current, directories, filenames in os.walk(ARTIFACT_ROOT, followlinks=False, onerror=walk_error):
        current_path = Path(current)
        directories[:] = sorted(name for name in directories if not (current_path / name).is_symlink())
        for filename in sorted(filenames):
            path = current_path / filename
            if path.is_symlink():
                if path.suffix.lower() == '.json':
                    add_blocker(f'symlink JSON artifact rejected: {path}')
                continue
            if path.is_file():
                all_files.append(path.resolve())
    for name in STABLE_ARTIFACTS:
        matches = sorted(path for path in all_files if path.name == name)
        ARTIFACT_MATCHES[name] = matches
        if len(matches) > 1:
            add_blocker(f'ambiguous artifact name: {name}')
        elif len(matches) == 1:
            ARTIFACT_PATHS[name] = matches[0]
    for path in sorted({item for item in all_files if item.suffix.lower() == '.json'}):
        try:
            strict_json_load(path)
        except (OSError, ValueError) as exc:
            add_blocker(f'strict JSON rejected: {path}: {exc}')
    if 'tasks.json' in ARTIFACT_PATHS and 'public_tasks.json' in ARTIFACT_PATHS:
        add_blocker('both tasks.json and public_tasks.json are present; choose one')
    if 'tasks.json' not in ARTIFACT_PATHS and 'public_tasks.json' in ARTIFACT_PATHS:
        ARTIFACT_PATHS['tasks.json'] = ARTIFACT_PATHS['public_tasks.json']
    if REGISTRY_OVERRIDE is None and 'registry.json' in ARTIFACT_PATHS:
        REGISTRY_OVERRIDE = ARTIFACT_PATHS['registry.json']
    if REGISTRY_OVERRIDE is None and SOURCE_ROOT is not None:
        REGISTRY_OVERRIDE = SOURCE_ROOT / 'x_factor' / 'registry' / 'checkpoints.json'
else:
    add_blocker('X1_ARTIFACT_ROOT is not configured or does not exist')

runtime_report: dict[str, Any] = {
    'schema': 'x1-real-1-colab-cuda-runtime/v1',
    'python_version': platform.python_version(),
    'platform': platform.platform(),
    'device_requested': DEVICE,
    'cuda_available': False,
    'cuda_device_count': 0,
    'gpu_devices': [],
    'disk': {},
    'model_execution': False,
    'training': False,
}
try:
    runtime_report['disk'] = {
        'total_bytes': shutil.disk_usage(CONTENT_ROOT).total,
        'free_bytes': shutil.disk_usage(CONTENT_ROOT).free,
    }
except OSError as exc:
    add_blocker(f'disk probe failed: {exc}')
try:
    torch_spec = importlib.util.find_spec('torch')
except (ImportError, ValueError):
    torch_spec = None
if torch_spec is None:
    add_blocker('PyTorch is unavailable; select a CUDA Colab runtime')
else:
    try:
        torch_module = importlib.import_module('torch')
        runtime_report['torch_version'] = getattr(torch_module, '__version__', None)
        runtime_report['cuda_available'] = bool(torch_module.cuda.is_available())
        runtime_report['cuda_device_count'] = int(torch_module.cuda.device_count()) if runtime_report['cuda_available'] else 0
        if not runtime_report['cuda_available']:
            add_blocker('CUDA is unavailable; this notebook does not fall back to CPU or TPU')
    except Exception as exc:
        add_blocker(f'CUDA runtime probe failed: {type(exc).__name__}: {exc}')
nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi:
    try:
        probe = subprocess.run([nvidia_smi, '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader,nounits'], capture_output=True, text=True, check=False, timeout=20)
        if probe.returncode == 0:
            runtime_report['gpu_devices'] = [{'record': line.strip()} for line in probe.stdout.splitlines() if line.strip()]
    except (OSError, subprocess.SubprocessError):
        pass
if SOURCE_ROOT is not None:
    runtime_report['source_root'] = str(SOURCE_ROOT)
    runtime_report['source_runner_sha256'] = sha256_file(RUNNER_SOURCE) if RUNNER_SOURCE is not None and RUNNER_SOURCE.is_file() else None
    runtime_report['protocol_path'] = str(PROTOCOL_PATH) if PROTOCOL_PATH is not None else None
runtime_report['blockers'] = list(BLOCKERS)
RUNTIME_REPORT, RUNTIME_RESUMED = persist_or_load(RUN_ROOT / 'runtime.json', runtime_report)
print(json.dumps({'runtime': RUNTIME_REPORT, 'resumed': RUNTIME_RESUMED, 'source_root': str(SOURCE_ROOT) if SOURCE_ROOT else None, 'artifacts': {key: str(value) for key, value in ARTIFACT_PATHS.items()}, 'blockers': BLOCKERS}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import subprocess
import sys

CLI_RESULTS: dict[str, dict[str, Any]] = {}


def cli_timeout_seconds() -> int:
    raw = os.environ.get('X1_CLI_TIMEOUT_SECONDS', '10800').strip()
    try:
        value = int(raw)
    except ValueError:
        value = 10800
    return value if 60 <= value <= 43200 else 10800

def run_coordinator_cli(name: str, arguments: list[str], accepted: tuple[int, ...], expected_statuses: tuple[str, ...] = (), module_name: str = 'x_factor.x1_real_1') -> dict[str, Any]:
    if SOURCE_ROOT is None:
        return {'name': name, 'status': 'BLOCKED', 'returncode': None, 'blockers': ['SOURCE_UNAVAILABLE']}
    receipt_path = RUN_ROOT / f'{name}.json'
    cli_dir = RUN_ROOT / 'cli'
    cli_dir.mkdir(parents=True, exist_ok=True)
    stdout_path = cli_dir / f'{name}.stdout.txt'
    stderr_path = cli_dir / f'{name}.stderr.txt'
    execution_path = cli_dir / f'{name}.execution.json'
    command = [sys.executable, '-m', module_name, *map(str, arguments)]
    if receipt_path.exists():
        receipt = strict_json_load(receipt_path)
        return {'name': name, 'status': receipt.get('status'), 'valid': receipt.get('valid'), 'returncode': 0, 'receipt': receipt, 'resumed': True, 'command': command}
    if stdout_path.exists() or stderr_path.exists() or execution_path.exists():
        return {'name': name, 'status': 'BLOCKED', 'returncode': None, 'blockers': ['PARTIAL_CLI_STATE_EXISTS'], 'resumed': True, 'command': command}
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(SOURCE_ROOT) + (os.pathsep + environment['PYTHONPATH'] if environment.get('PYTHONPATH') else '')
    try:
        process = subprocess.run(command, cwd=str(SOURCE_ROOT), env=environment, capture_output=True, text=True, check=False, timeout=cli_timeout_seconds())
        returncode = process.returncode
        stdout = process.stdout
        stderr = process.stderr
        error = None
    except subprocess.TimeoutExpired as exc:
        returncode = None
        stdout = exc.stdout.decode('utf-8', errors='replace') if isinstance(exc.stdout, bytes) else (exc.stdout or '')
        stderr = exc.stderr.decode('utf-8', errors='replace') if isinstance(exc.stderr, bytes) else (exc.stderr or '')
        error = 'timeout'
    stdout_path.write_text(stdout, encoding='utf-8', newline='\n')
    stderr_path.write_text(stderr, encoding='utf-8', newline='\n')
    execution = {'schema': 'x1-real-1-colab-cuda-cli-execution/v1', 'command': command, 'returncode': returncode, 'error': error}
    write_json_no_overwrite(execution_path, execution)
    try:
        parsed = strict_json_loads(stdout, str(stdout_path))
    except (OSError, ValueError) as exc:
        parsed = {'schema': 'x1-real-1-colab-cuda-cli-parse-error/v1', 'status': 'ERROR', 'error': str(exc)}
    write_json_no_overwrite(receipt_path, parsed)
    result = {'name': name, 'status': parsed.get('status') if isinstance(parsed, dict) else None, 'valid': parsed.get('valid') if isinstance(parsed, dict) else None, 'returncode': returncode, 'receipt': parsed, 'resumed': False, 'command': command, 'expected_blocked_result': returncode == 2 and isinstance(parsed, dict) and parsed.get('status') in expected_statuses}
    return result

if SOURCE_ROOT is not None and PROTOCOL_PATH is not None:
    CLI_RESULTS['protocol'] = run_coordinator_cli('protocol_validation', ['validate-protocol', '--protocol', str(PROTOCOL_PATH), '--repo-root', str(SOURCE_ROOT), '--check-source-closure'], (0, 2), ('PROTOCOL_INVALID',))
    if REGISTRY_OVERRIDE is not None:
        CLI_RESULTS['inventory'] = run_coordinator_cli('checkpoint_inventory', ['inventory', '--registry', str(REGISTRY_OVERRIDE), '--verify-files'], (0, 2), ('NO_ELIGIBLE_CHECKPOINT', 'ELIGIBLE_SUBJECT_AVAILABLE'))
        gate_args = ['gate', '--protocol', str(PROTOCOL_PATH), '--registry', str(REGISTRY_OVERRIDE)]
        if 'basis_qualification.json' in ARTIFACT_PATHS:
            gate_args.extend(['--basis-qualification', str(ARTIFACT_PATHS['basis_qualification.json'])])
        if 'release_manifest.json' in ARTIFACT_PATHS:
            gate_args.extend(['--release-manifest', str(ARTIFACT_PATHS['release_manifest.json'])])
        if RELEASE_ROOT is not None:
            gate_args.extend(['--release-root', str(RELEASE_ROOT)])
        CLI_RESULTS['gate'] = run_coordinator_cli('prerequisite_gate', gate_args, (0, 2), ('NO_ELIGIBLE_CHECKPOINT', 'BASIS_NOT_QUALIFIED', 'RELEASE_MANIFEST_REQUIRED', 'READY_FOR_EXTERNAL_PHASE_1', 'PROTOCOL_INVALID'))
        preflight_args = ['preflight', '--protocol', str(PROTOCOL_PATH), '--source-root', str(SOURCE_ROOT), '--out', str(RUN_ROOT / 'preflight.json')]
        options = {'subject_manifest.json': '--subject', 'basis_qualification.json': '--basis-qualification', 'release_manifest.json': '--release-manifest', 'primary_split.json': '--split', 'tasks.json': '--tasks', 'basis_split.json': '--basis-split', 'development_split.json': '--development-split', 'prediction_receipt.json': '--prediction', 'prediction_commit.json': '--prediction-commit'}
        for name, option in options.items():
            if name in ARTIFACT_PATHS:
                preflight_args.extend([option, str(ARTIFACT_PATHS[name])])
        preflight_args.extend(['--registry', str(REGISTRY_OVERRIDE)])
        if CHECKPOINT_ROOT is not None:
            preflight_args.extend(['--checkpoint-root', str(CHECKPOINT_ROOT)])
        if RELEASE_ROOT is not None:
            preflight_args.extend(['--release-root', str(RELEASE_ROOT)])
        CLI_RESULTS['preflight'] = run_coordinator_cli('preflight', preflight_args, (0, 2), ('BLOCKED', 'READY_FOR_EXTERNAL_PREDICTION'))
        if (RUN_ROOT / 'preflight.json').exists():
            plan_args = ['execution-plan', '--protocol', str(PROTOCOL_PATH), '--preflight', str(RUN_ROOT / 'preflight.json'), '--source-root', str(SOURCE_ROOT), '--out', str(RUN_ROOT / 'execution_plan.json')]
            CLI_RESULTS['execution_plan'] = run_coordinator_cli('execution_plan', plan_args, (0, 2), ('BLOCKED', 'BLOCKED_POWER_DEVIATION_REQUIRED', 'READY_FOR_EXTERNAL_EXECUTION'))
    else:
        add_blocker('checkpoint registry is unavailable')
else:
    add_blocker('source/protocol validation is unavailable')
write_json_no_overwrite(RUN_ROOT / 'gate_summary.json', {'schema': 'x1-real-1-colab-cuda-gates/v1', 'results': CLI_RESULTS, 'blockers': BLOCKERS})
print(json.dumps({name: {'returncode': result.get('returncode'), 'status': result.get('status'), 'valid': result.get('valid'), 'blockers': result.get('blockers', [])} for name, result in CLI_RESULTS.items()}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import os

RUN_CUDA_INTAKE = os.environ.get('RUN_CUDA_INTAKE', '1').strip() == '1'
INTAKE_RESULT: dict[str, Any]

if not RUN_CUDA_INTAKE:
    INTAKE_RESULT = {'schema': 'x1-real-1-colab-cuda-intake-status/v1', 'status': 'NOT_REQUESTED', 'executed': False, 'model_execution': False, 'training': False, 'next_action': 'set RUN_CUDA_INTAKE=1 to hash an attached candidate without loading it'}
elif SOURCE_ROOT is None or CHECKPOINT_PATH is None or not CHECKPOINT_PATH.is_file():
    INTAKE_RESULT = {'schema': 'x1-real-1-colab-cuda-intake-status/v1', 'status': 'BLOCKED', 'executed': False, 'model_execution': False, 'training': False, 'blockers': ['INTAKE_SOURCE_OR_CHECKPOINT_UNAVAILABLE']}
else:
    intake_args = ['intake', '--checkpoint', str(CHECKPOINT_PATH)]
    optional_arguments = {
        '--config': CONFIG_PATH,
        '--tokenizer': TOKENIZER_PATH,
        '--runtime-source-revision': os.environ.get('X1_RUNTIME_SOURCE_REVISION'),
        '--source-commit': os.environ.get('X1_SOURCE_COMMIT'),
        '--global-step': os.environ.get('X1_GLOBAL_STEP'),
        '--stage': os.environ.get('X1_STAGE'),
        '--parameter-sha256': os.environ.get('X1_PARAMETER_SHA256'),
        '--tokenizer-identity-sha256': os.environ.get('X1_TOKENIZER_IDENTITY_SHA256'),
        '--readiness-receipt': os.environ.get('X1_READINESS_RECEIPT'),
        '--readiness-receipt-sha256': os.environ.get('X1_READINESS_RECEIPT_SHA256'),
    }
    for option, value in optional_arguments.items():
        if value:
            intake_args.extend([option, str(value)])
    intake_args.extend(['--out', str(RUN_ROOT / 'candidate_intake.json')])
    INTAKE_RESULT = run_coordinator_cli('candidate_intake', intake_args, (0, 2), ('INVENTORIED', 'UNQUALIFIED_NEW'))
persist_intake, intake_resumed = persist_or_load(RUN_ROOT / 'candidate_intake_status.json', INTAKE_RESULT)
INTAKE_RESULT = persist_intake
print(json.dumps({'intake': INTAKE_RESULT, 'resumed': intake_resumed, 'output': str(RUN_ROOT / 'candidate_intake.json')}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
import subprocess
import sys

RUN_CUDA_CANARY = os.environ.get('RUN_CUDA_CANARY', '0').strip() == '1'
CANARY_AUTHORIZATION_PHRASE = 'I AUTHORIZE X1-REAL-1 CUDA CANARY'
CANARY_OUTPUT = RUN_ROOT / 'cuda-canary.json'
CANARY_INTAKE_PATH = ARTIFACT_PATHS.get('candidate_intake.json') or (RUN_ROOT / 'candidate_intake.json' if (RUN_ROOT / 'candidate_intake.json').is_file() else None)
CANARY_RESULT: dict[str, Any]

if not RUN_CUDA_CANARY:
    CANARY_RESULT = {'schema': 'x1-real-1-colab-cuda-canary-status/v1', 'status': 'NOT_REQUESTED', 'executed': False, 'scientific_subject': False, 'scientific_completion_claimed': False, 'training': False, 'next_action': 'set RUN_CUDA_CANARY=1 only for an explicitly authorized non-scientific runtime check'}
elif SOURCE_ROOT is None or not RUNTIME_REPORT.get('cuda_available'):
    CANARY_RESULT = {'schema': 'x1-real-1-colab-cuda-canary-status/v1', 'status': 'BLOCKED', 'executed': False, 'scientific_subject': False, 'scientific_completion_claimed': False, 'training': False, 'blockers': ['CUDA_SOURCE_OR_RUNTIME_UNAVAILABLE']}
elif CHECKPOINT_PATH is None or CONFIG_PATH is None or TOKENIZER_PATH is None or CANARY_INTAKE_PATH is None:
    CANARY_RESULT = {'schema': 'x1-real-1-colab-cuda-canary-status/v1', 'status': 'BLOCKED', 'executed': False, 'scientific_subject': False, 'scientific_completion_claimed': False, 'training': False, 'blockers': ['CANARY_CHECKPOINT_CONFIG_TOKENIZER_OR_INTAKE_MISSING']}
elif os.environ.get('X1_CANARY_AUTHORIZATION', '') != CANARY_AUTHORIZATION_PHRASE:
    CANARY_RESULT = {'schema': 'x1-real-1-colab-cuda-canary-status/v1', 'status': 'BLOCKED', 'executed': False, 'scientific_subject': False, 'scientific_completion_claimed': False, 'training': False, 'blockers': ['EXACT_CANARY_AUTHORIZATION_PHRASE_REQUIRED']}
elif CANARY_OUTPUT.exists():
    try:
        CANARY_RESULT = strict_json_load(CANARY_OUTPUT)
    except (OSError, ValueError) as exc:
        CANARY_RESULT = {'schema': 'x1-real-1-colab-cuda-canary-status/v1', 'status': 'BLOCKED', 'executed': False, 'scientific_subject': False, 'scientific_completion_claimed': False, 'training': False, 'blockers': [f'INVALID_EXISTING_CANARY:{exc}']}
else:
    canary_args = ['--checkpoint', str(CHECKPOINT_PATH), '--config', str(CONFIG_PATH), '--tokenizer', str(TOKENIZER_PATH), '--intake', str(CANARY_INTAKE_PATH), '--output', str(CANARY_OUTPUT), '--source-root', str(SOURCE_ROOT), '--device', DEVICE, '--max-new-tokens', '1', '--authorization', CANARY_AUTHORIZATION_PHRASE]
    canary_command = [sys.executable, '-m', 'x_factor.x1_cuda_canary', *canary_args]
    canary_dir = RUN_ROOT / 'canary-cli'
    canary_dir.mkdir(parents=True, exist_ok=True)
    canary_stdout = canary_dir / 'stdout.txt'
    canary_stderr = canary_dir / 'stderr.txt'
    canary_execution = canary_dir / 'execution.json'
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(SOURCE_ROOT) + (os.pathsep + environment['PYTHONPATH'] if environment.get('PYTHONPATH') else '')
    try:
        process = subprocess.run(canary_command, cwd=str(SOURCE_ROOT), env=environment, capture_output=True, text=True, check=False, timeout=1800)
        returncode = process.returncode
        stdout = process.stdout
        stderr = process.stderr
        error = None
    except subprocess.TimeoutExpired as exc:
        returncode = None
        stdout = exc.stdout.decode('utf-8', errors='replace') if isinstance(exc.stdout, bytes) else (exc.stdout or '')
        stderr = exc.stderr.decode('utf-8', errors='replace') if isinstance(exc.stderr, bytes) else (exc.stderr or '')
        error = 'timeout'
    write_json_no_overwrite(canary_execution, {'schema': 'x1-real-1-colab-cuda-canary-execution/v1', 'command': canary_command, 'returncode': returncode, 'error': error})
    canary_stdout.write_text(stdout, encoding='utf-8', newline='\n')
    canary_stderr.write_text(stderr, encoding='utf-8', newline='\n')
    try:
        parsed = strict_json_loads(stdout, str(canary_stdout))
    except (OSError, ValueError) as exc:
        parsed = {'schema': 'x1-real-1-colab-cuda-canary-parse-error/v1', 'status': 'ERROR', 'error': str(exc)}
    if returncode == 0 and CANARY_OUTPUT.exists():
        CANARY_RESULT = strict_json_load(CANARY_OUTPUT)
    else:
        CANARY_RESULT = {'schema': 'x1-real-1-colab-cuda-canary-status/v1', 'status': 'BLOCKED', 'executed': False, 'scientific_subject': False, 'scientific_completion_claimed': False, 'training': False, 'runner_result': parsed, 'returncode': returncode, 'blockers': ['CANARY_PROCESS_DID_NOT_PRODUCE_A_VALID_RECEIPT']}
persisted_canary, canary_resumed = persist_or_load(RUN_ROOT / 'cuda_canary_status.json', CANARY_RESULT)
CANARY_RESULT = persisted_canary
print(json.dumps({'intake': INTAKE_RESULT, 'canary': CANARY_RESULT, 'resumed': canary_resumed, 'output': str(CANARY_OUTPUT)}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path

RUN_EXTERNAL_CUDA = os.environ.get('RUN_EXTERNAL_CUDA', '0').strip() == '1'
EXECUTION_OUTPUT = RUN_ROOT / 'external-run'
EXECUTION_RESULT: dict[str, Any]

if not RUN_EXTERNAL_CUDA:
    EXECUTION_RESULT = {'schema': 'x1-real-1-colab-cuda-execution-status/v1', 'status': 'NOT_REQUESTED', 'executed': False, 'scientific_completion_claimed': False, 'training': False, 'next_action': 'set RUN_EXTERNAL_CUDA=1 only after all gates and the exact authorization phrase pass'}
    persist_or_load(RUN_ROOT / 'external_execution_status.json', EXECUTION_RESULT)
else:
    required_names = ['preflight.json', 'subject_manifest.json', 'registry.json', 'release_manifest.json', 'primary_split.json', 'tasks.json', 'basis_qualification.json', 'basis_split.json', 'development_split.json', 'prediction_receipt.json', 'prediction_commit.json']
    missing = [name for name in required_names if name not in ARTIFACT_PATHS]
    required_files = {'checkpoint': CHECKPOINT_PATH, 'config': CONFIG_PATH, 'tokenizer': TOKENIZER_PATH}
    missing.extend(f'{name}:{value}' for name, value in required_files.items() if value is None or not value.is_file())
    if BLOCKERS or missing or SOURCE_ROOT is None:
        EXECUTION_RESULT = {'schema': 'x1-real-1-colab-cuda-execution-status/v1', 'status': 'BLOCKED', 'executed': False, 'scientific_completion_claimed': False, 'training': False, 'missing_artifacts': missing, 'blockers': list(BLOCKERS), 'next_action': 'resolve every source, CUDA, artifact, and gate blocker before external execution'}
    elif os.environ.get('X1_AUTHORIZATION', '') != AUTHORIZATION_PHRASE:
        EXECUTION_RESULT = {'schema': 'x1-real-1-colab-cuda-execution-status/v1', 'status': 'BLOCKED', 'executed': False, 'scientific_completion_claimed': False, 'training': False, 'blockers': ['EXACT_AUTHORIZATION_PHRASE_REQUIRED'], 'next_action': 'operator must set X1_AUTHORIZATION to the exact phrase'}
    else:
        args = ['--protocol', str(PROTOCOL_PATH), '--preflight', str(ARTIFACT_PATHS['preflight.json']), '--subject', str(ARTIFACT_PATHS['subject_manifest.json']), '--registry', str(ARTIFACT_PATHS['registry.json']), '--release-manifest', str(ARTIFACT_PATHS['release_manifest.json']), '--split', str(ARTIFACT_PATHS['primary_split.json']), '--public-tasks', str(ARTIFACT_PATHS['tasks.json']), '--basis-qualification', str(ARTIFACT_PATHS['basis_qualification.json']), '--basis-split', str(ARTIFACT_PATHS['basis_split.json']), '--development-split', str(ARTIFACT_PATHS['development_split.json']), '--prediction', str(ARTIFACT_PATHS['prediction_receipt.json']), '--prediction-commit', str(ARTIFACT_PATHS['prediction_commit.json']), '--checkpoint', str(CHECKPOINT_PATH), '--config', str(CONFIG_PATH), '--tokenizer', str(TOKENIZER_PATH), '--output', str(EXECUTION_OUTPUT), '--authorization', AUTHORIZATION_PHRASE, '--device', DEVICE, '--max-new-tokens', str(MAX_NEW_TOKENS), '--source-root', str(SOURCE_ROOT), '--release-root', str(RELEASE_ROOT or SOURCE_ROOT), '--checkpoint-root', str(CHECKPOINT_ROOT or (CHECKPOINT_PATH.parent if CHECKPOINT_PATH else SOURCE_ROOT))]
        try:
            execution_label = 'external_cuda_runner_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
            EXECUTION_RESULT = run_coordinator_cli(execution_label, args, (0,), ('RAW_EXTERNAL_EXECUTION_COMPLETE',), module_name='x_factor.x1_external_runner')
            if EXECUTION_RESULT.get('returncode') != 0 or EXECUTION_RESULT.get('status') != 'RAW_EXTERNAL_EXECUTION_COMPLETE':
                EXECUTION_RESULT = {'schema': 'x1-real-1-colab-cuda-execution-status/v1', 'status': 'BLOCKED', 'executed': False, 'scientific_completion_claimed': False, 'training': False, 'runner_result': EXECUTION_RESULT, 'next_action': 'inspect the captured runner stderr/JSON and correct the external evidence; no fallback is permitted'}
        except Exception as exc:
            EXECUTION_RESULT = {'schema': 'x1-real-1-colab-cuda-execution-status/v1', 'status': 'BLOCKED', 'executed': False, 'scientific_completion_claimed': False, 'training': False, 'error': f'{type(exc).__name__}: {exc}', 'next_action': 'correct the CUDA runner inputs and rerun with the same output directory'}
    persisted, resumed = persist_or_load(RUN_ROOT / 'external_execution_status.json', EXECUTION_RESULT)
    EXECUTION_RESULT = persisted
    print(json.dumps({'intake': INTAKE_RESULT,
    'execution': EXECUTION_RESULT,
    'canary': CANARY_RESULT, 'resumed': resumed, 'output': str(EXECUTION_OUTPUT)}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from pathlib import Path
import zipfile

for result in CLI_RESULTS.values():
    receipt = result.get('receipt') if isinstance(result, dict) else None
    if isinstance(receipt, dict):
        for blocker in receipt.get('blockers', []):
            add_blocker(str(blocker))
        if receipt.get('status') in {'NO_ELIGIBLE_CHECKPOINT', 'BASIS_NOT_QUALIFIED', 'RELEASE_MANIFEST_REQUIRED', 'PROTOCOL_INVALID'}:
            add_blocker(str(receipt.get('status')))
if INTAKE_RESULT.get('status') == 'BLOCKED':
    for blocker in INTAKE_RESULT.get('blockers', ['INTAKE_BLOCKED']):
        add_blocker(str(blocker))
if CANARY_RESULT.get('status') == 'BLOCKED':
    for blocker in CANARY_RESULT.get('blockers', ['CANARY_BLOCKED']):
        add_blocker(str(blocker))
if EXECUTION_RESULT.get('status') == 'BLOCKED':
    add_blocker('SCIENTIFIC_EXTERNAL_EXECUTION_BLOCKED')
FINAL_STATUS = {
    'schema': 'x1-real-1-colab-cuda-final-status/v1',
    'status': 'RAW_EXTERNAL_EXECUTION_REPORTED' if EXECUTION_RESULT.get('status') == 'RAW_EXTERNAL_EXECUTION_COMPLETE' else 'BLOCKED',
    'scientific_completion_claimed': False,
    'raw_external_execution_only': True,
    'training': False,
    'weight_updates': False,
    'protocol': str(PROTOCOL_PATH) if PROTOCOL_PATH else None,
    'protocol_sha256': CLI_RESULTS.get('protocol', {}).get('receipt', {}).get('protocol_sha256') if isinstance(CLI_RESULTS.get('protocol'), dict) else None,
    'runtime': RUNTIME_REPORT,
    'intake': INTAKE_RESULT,
    'execution': EXECUTION_RESULT,
    'canary': CANARY_RESULT,
    'blockers': list(BLOCKERS),
    'limitations': [
        'no outcome labels or verifier artifacts',
        'no reveal receipt or analysis',
        'no predictor fitting or basis qualification',
        'no scientific completion claim',
    ],
}
FINAL_STATUS, FINAL_RESUMED = persist_or_load(RUN_ROOT / 'final_status.json', FINAL_STATUS)
archive = CONTENT_ROOT / f'x1-real-1-cuda-{RUN_ID}-receipts.zip'
partial = CONTENT_ROOT / f'.x1-real-1-cuda-{RUN_ID}-receipts.partial.zip'
receipt_path = CONTENT_ROOT / f'x1-real-1-cuda-{RUN_ID}-receipts.zip.sha256.json'
if not archive.exists():
    if partial.exists():
        raise RuntimeError(f'partial archive already exists and will not be overwritten: {partial}')
    members = []
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_symlink():
            raise RuntimeError(f'run root contains a symlink: {path}')
        if path.is_file():
            members.append((path, path.relative_to(RUN_ROOT).as_posix()))
    with zipfile.ZipFile(partial, 'x', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as bundle:
        for path, member in members:
            bundle.write(path, arcname=f'{RUN_ID}/{member}')
    with zipfile.ZipFile(partial, 'r') as bundle:
        if bundle.testzip() is not None:
            raise RuntimeError('receipt archive failed CRC verification')
    try:
        os.link(partial, archive)
        partial.unlink()
    except OSError:
        if partial.exists() and not archive.exists():
            os.replace(partial, archive)
        else:
            raise
digest = sha256_file(archive)
archive_receipt = {'schema': 'x1-real-1-colab-cuda-archive/v1', 'status': 'VERIFIED', 'archive': archive.name, 'sha256': digest, 'bytes': archive.stat().st_size, 'run_id': RUN_ID, 'checkpoint_weights_included': False, 'scientific_completion_claimed': False, 'training': False}
if receipt_path.exists():
    existing = strict_json_load(receipt_path)
    if existing != archive_receipt:
        raise RuntimeError('existing archive receipt differs from the current archive')
else:
    write_json_no_overwrite(receipt_path, archive_receipt)
print(json.dumps({'final_status': FINAL_STATUS, 'final_resumed': FINAL_RESUMED, 'archive': str(archive), 'sha256': digest, 'receipt': str(receipt_path)}, indent=2, sort_keys=True))
try:
    from IPython.display import FileLink, display
    display(FileLink(str(archive)))
except Exception:
    print('IPython FileLink is unavailable; download the archive from the Colab file pane.')


## What this notebook does not do

A successful canary receipt proves only strict CUDA runtime/checkpoint compatibility. A successful `RAW_EXTERNAL_EXECUTION_COMPLETE` manifest proves only that the strict CUDA runtime executed the supplied task/intervention matrix and wrote immutable raw receipts. It does not establish correctness, task competence, intervention effects, or a scientific X1 result. Outcome labeling, gold isolation, verifier artifact creation, reveal commitment, analysis, replication, and any predictor or basis work require separately released, file-bound producers and custody operators.

If the gate cell reports `BLOCKED`, do not bypass it by editing the protocol, promoting the historical checkpoint, changing the device, or deleting receipts. Resolve the reported identity, release, basis, split, prediction, storage, or CUDA blocker and rerun with the same immutable run root.
